In [6]:
import pandas as pd
from itertools import combinations

# Create the transaction data
transactions = [
    ['Bread', 'Butter', 'Milk'],  # T1
    ['Bread', 'Butter'],          # T2
    ['Bread', 'Milk'],            # T3
    ['Butter', 'Milk'],           # T4
    ['Bread', 'Milk']             # T5
]

# Define the minimum support and confidence thresholds
MIN_SUPPORT = 0.5 # 40% of transactions
MIN_CONFIDENCE = 0.7  # 60% confidence

# Number of transactions
num_transactions = len(transactions)
print(f"Total transactions: {num_transactions}")

# Step 1: Find all unique items
unique_items = sorted(list(set(item for transaction in transactions for item in transaction)))
print(f"\nUnique items: {unique_items}")

# Step 2: Create one-hot encoded representation for visualization
one_hot = pd.DataFrame(
    [[1 if item in transaction else 0 for item in unique_items] for transaction in transactions],
    columns=unique_items,
    index=[f'T{i+1}' for i in range(len(transactions))]
)
print("\nOne-hot encoded transaction data:")
print(one_hot)

# Step 3: Calculate support for individual items
item_support = {}
for item in unique_items:
    count = sum(1 for t in transactions if item in t)
    support = count / num_transactions
    item_support[item] = (count, support)

print("\nIndividual item support:")
for item, (count, support) in item_support.items():
    print(f"{item}: {count}/{num_transactions} = {support:.2f}")

# Step 4: Find frequent individual items (meeting minimum support)
frequent_items = [item for item, (_, support) in item_support.items() if support >= MIN_SUPPORT]
print(f"\nFrequent items (support >= {MIN_SUPPORT}):")
print(frequent_items)

# Step 5: Generate frequent itemsets dictionary (will contain both individual items and pairs)
frequent_itemsets = {
    frozenset([item]): item_support[item][1] for item in frequent_items
}

# Step 6: Find frequent pairs
for item_pair in combinations(frequent_items, 2):
    pair_set = frozenset(item_pair)
    pair_count = sum(1 for t in transactions if all(item in t for item in pair_set))
    pair_support = pair_count / num_transactions
    
    if pair_support >= MIN_SUPPORT:
        print(f"\nPair {set(pair_set)} appears in {pair_count}/{num_transactions} transactions (support = {pair_support:.2f})")
        frequent_itemsets[pair_set] = pair_support

print("\nAll frequent itemsets with support >= 0.4:")
for itemset, support in frequent_itemsets.items():
    print(f"{set(itemset)}: {support:.2f}")

# Step 7: Generate association rules
rules = []
for itemset, itemset_support in frequent_itemsets.items():
    if len(itemset) > 1:  # Only generate rules from pairs
        for item in itemset:
            antecedent = frozenset([item])
            consequent = itemset - antecedent
            
            antecedent_support = frequent_itemsets[antecedent]
            confidence = itemset_support / antecedent_support
            
            if confidence >= MIN_CONFIDENCE:
                # Calculate lift
                consequent_support = frequent_itemsets[frozenset(consequent)]
                lift = confidence / consequent_support
                
                rules.append((antecedent, consequent, itemset_support, confidence, lift))

print(f"\nAssociation rules (confidence >= {MIN_CONFIDENCE}):")
for antecedent, consequent, support, confidence, lift in rules:
    print(f"{set(antecedent)} => {set(consequent)}: support={support:.2f}, confidence={confidence:.2f}, lift={lift:.2f}")

# Interpret the results
print("\n--- INTERPRETATION ---")
print("The strongest association rules found:")
for antecedent, consequent, support, confidence, lift in sorted(rules, key=lambda x: x[3], reverse=True):
    ant = list(antecedent)[0]
    cons = list(consequent)[0]
    print(f"- When customers buy {ant}, they buy {cons} with {confidence:.0%} confidence")
    print(f"  (This rule has support={support:.2f} and lift={lift:.2f})")

Total transactions: 5

Unique items: ['Bread', 'Butter', 'Milk']

One-hot encoded transaction data:
    Bread  Butter  Milk
T1      1       1     1
T2      1       1     0
T3      1       0     1
T4      0       1     1
T5      1       0     1

Individual item support:
Bread: 4/5 = 0.80
Butter: 3/5 = 0.60
Milk: 4/5 = 0.80

Frequent items (support >= 0.5):
['Bread', 'Butter', 'Milk']

Pair {'Milk', 'Bread'} appears in 3/5 transactions (support = 0.60)

All frequent itemsets with support >= 0.4:
{'Bread'}: 0.80
{'Butter'}: 0.60
{'Milk'}: 0.80
{'Milk', 'Bread'}: 0.60

Association rules (confidence >= 0.7):
{'Milk'} => {'Bread'}: support=0.60, confidence=0.75, lift=0.94
{'Bread'} => {'Milk'}: support=0.60, confidence=0.75, lift=0.94

--- INTERPRETATION ---
The strongest association rules found:
- When customers buy Milk, they buy Bread with 75% confidence
  (This rule has support=0.60 and lift=0.94)
- When customers buy Bread, they buy Milk with 75% confidence
  (This rule has support=0.60